In [1]:
# 1. Environment & Path Setup
import os
import sys

# Add the project root directory to sys.path so 'pipeline' is importable
sys.path.append(os.path.abspath('..'))

import numpy as np  # noqa: F401
import pandas as pd

from src.database import get_matchevent_receptions, get_matchevent_releases
from src.features import get_normalised_y, op_pm_avg_locations

In [2]:
match_id = 3754217
team_id = 1

In [3]:
rec_df = get_matchevent_receptions(team_id, match_id)

In [4]:
rel_df = get_matchevent_releases(team_id, match_id)

In [5]:
rec_df = get_normalised_y(rec_df)

In [6]:
rel_df = get_normalised_y(rel_df)

In [7]:
OPEN_PLAY = ['Regular Play', 'From Keeper', 'From Counter', 'From Throw In']
op_rec = rec_df[rec_df["play_pattern"].isin(OPEN_PLAY)]
op_rel = rel_df[rel_df["play_pattern"].isin(OPEN_PLAY)]

In [8]:
av_x_op_rec = op_rec.groupby("player_id")["x"].mean()
av_norm_y_op_rec = op_rec.groupby("player_id")["normalised_y"].mean()

In [9]:
av_x_op_rel = op_rel.groupby("player_id")["x"].mean()
av_norm_y_op_rel = op_rel.groupby("player_id")["normalised_y"].mean()

In [10]:
KEYS = ["average_reception_x", "average_reception_norm_y", "average_release_x", "average_release_norm_y"]
av_loc = pd.concat([av_x_op_rec, av_norm_y_op_rec, av_x_op_rel, av_norm_y_op_rel], axis = 1, keys = KEYS)

In [11]:
rec_groupby = op_rec.groupby("player_id")
rel_groupby = op_rel.groupby("player_id")

In [12]:
test_rec_gb = rec_groupby.agg(
    team_id = ("team_id", "first"),
    match_id = ("match_id", "first"),
    player = ("player", "first"),
    total_reception_events=("player_id", "size"),
    average_reception_x=("x", "mean"),
    average_reception_norm_y=("normalised_y", "mean")
)

In [13]:
test_rel_gb = rel_groupby.agg(
    total_release_events = ("player_id", "size"),
    average_release_x = ("x", "mean"),
    average_release_norm_y = ("y", "mean")
)

In [14]:
test_gb = pd.concat([test_rec_gb, test_rel_gb], axis = 1)

In [15]:
test1_gb = op_pm_avg_locations(team_id, match_id)